# Trabajo Ev 1

## Leer DataSet


In [2]:
import pandas as pd
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

raw_df = pd.read_csv("retail_store_sales.csv")
#respaldo y copia del dataset original

df = raw_df
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


##Limpieza de | Price Per Unit | Quantity | Total Spent (Busqueda de Nulos)

In [3]:


#comprobamos la cantidad de campos nulos
df.isna().sum()


,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


Podemos concluir que  TODOS los registros tienen un "TRANSACTION_ID", "CUSTOMER_ID", "CATEGORY","PAYMENT METHOD",
"LOCATION" y "TRANSACTION DATE".

In [4]:
#comprobamos si existen transacciones repetidas
id_repetidos= raw_df['Transaction ID'].duplicated().sum()
#comprobamos si en general existen  registros repetidos (aunque si no hay transacciones repetidas esto se puede evitar)
datos_Repetidos = raw_df.duplicated().sum()

print(f"la cantidad de transacciones repetidas es: {id_repetidos}")
print(f"la cantidad de datos repetidos es: {datos_Repetidos}")

la cantidad de transacciones repetidas es: 0
la cantidad de datos repetidos es: 0


no existen errores de duplicación y cada transacción es unica

In [5]:
# En los siguientes 4 codigos se buscarán patrones en los que no se pueden recuperar los valores de las columnas "Price Per Unit"	"Quantity"	"Total Spent"
# Esto debido a que no por ejemplo , no podemos sacar el total sin la cantidad ni el precio unitario
#en general si hay menos de 2 campos de estas 3 columnas, el dato es irrecuperable.
# Busqueda de si "Precio", "Cantidad", "Total" son Nulos de manera simultanea

#Casos donde 'Price per unit', 'Quantity' y 'Total spent' son nulos
filtro_null_1 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_1.shape

(0, 3)

In [6]:
#Casos donde 'Cantidad' y 'Total' son Nulos
filtro_null_2 = df.loc[
      df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_2.shape

(604, 3)

In [7]:
#Casos donde 'Price per Unit y 'Quantity' son Nulos
filtro_null_3 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_3.shape

(0, 3)

In [8]:

#Casos donde 'Price per Unit y 'Total' son Nulos
filtro_null_4 =df.loc[
      df['Price Per Unit'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_4.shape

(0, 3)

Podemos observar que de los 4 escenarios posibles donde NO se pueden recuperar estos 3 datos, solo hay  604 escenarios donde no hay cantidad, ni precio unitario, en el resto de casos es recuperable la información por ende, solo limpiamos ese escenario.

In [9]:
#borramos los escenarios de 'Price per unit', 'Quantity' y 'Total spent' donde no es recuperable
df.drop(filtro_null_2.index, axis=0, inplace=True)
#revisamos los nulls denuevo
df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,609
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


##Recuperando campos vacios

Podemos observar del bloque de codigo anterior que los unicos campos nulos actuales entre 'Price Per Unit', 'Quantity' y 'Total Spent' solo hay 609 casos nulos de Price Per unit


In [11]:

#Buscaremos reparar los campos vacios de 'Price Per unit' donde NO hay descuento (ya que el descuento nos puede dar un valor sesgado del total real)

#PRICE PER UNIT NULL
desc_false = df['Discount Applied'] == False # filtramos las filas con descuento falso bajo el nombre 'desc_false '

df.loc[desc_false , 'Price Per Unit'] = df.loc[desc_false, 'Price Per Unit'].fillna(
    df['Total Spent'] / df['Quantity']
)

df.isna().sum()


,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,423
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0
